In [11]:
!pip install langchain chromadb langchain_groq tiktoken pypdf langchain-community langchain_huggingface langchain_chroma

In [3]:
import os
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
os.environ["GROQ_API_KEY"]="GROQ_KEY"
os.environ["HUGGINGFACEHUB_API_TOKEN"]="HUGGING_FACE_API_KEY"
os.environ["LANGSMITH_API_KEY"] = "LANG_SMITH_API_KEY"
os.environ["LANGSMITH_TRACING"] = "true"

In [6]:
from langchain_core.documents import Document

In [7]:
doc1 = Document(page_content='Virat Kohli is a world-renowned Indian international cricketer born on November 5, 1988, in Delhi. Considered one of the greatest batsmen in the history of the game, he is known for his aggressive style, exceptional run-chasing ability, and unwavering consistency, earning him the nickname "Chase Master" and "King Kohli".',
                metadata={"team": "RCB"})
doc2 = Document(page_content='Mahendra Singh Dhoni, affectionately known as "Mahi" or "Captain Cool," is a legendary Indian cricketer and one of the most successful captains in international cricket history',
                metadata={"team": "CSK"})
doc3 = Document(page_content='Rohit Gurunath Sharma (born 30 April 1987) is a legendary Indian international cricketer and former captain of the national team in all formats. Popularly nicknamed "Hitman," he is widely recognized for his elegant batting style, prolific six-hitting ability, and calm, astute leadership.',
                metadata={"team": "MI"})

In [15]:
all_documents = [doc1, doc2, doc3]
texts = [ doc.page_content for doc in all_documents ]

In [12]:
from langchain_chroma import Chroma

In [16]:
embedding_model = HuggingFaceEmbeddings(model='sentence-transformers/all-MiniLM-L6-v2')
embeddings = embedding_model.embed_documents(texts)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
len(embeddings)

3

In [21]:
vector_store = Chroma(
    embedding_function=embedding_model,
    persist_directory="./chroma_db",
    collection_name="sample"
)

In [22]:
vector_store.add_documents(all_documents)

['d85f78e3-c740-49fd-9fba-cc8bdaa40540',
 'ad701154-986a-4b18-a817-a93db0be28b1',
 '8b48442d-20ff-473a-b39e-f76dacfa343c']

In [27]:
vector_store.get(include=["embeddings", "documents", "metadatas"])

{'ids': ['d85f78e3-c740-49fd-9fba-cc8bdaa40540',
  'ad701154-986a-4b18-a817-a93db0be28b1',
  '8b48442d-20ff-473a-b39e-f76dacfa343c'],
 'embeddings': array([[ 1.63408853e-02,  1.05008014e-01, -5.56412600e-02, ...,
         -3.30899097e-02,  4.41147499e-02,  7.48309249e-05],
        [-4.31474596e-02,  3.62290144e-02,  1.87813374e-03, ...,
         -6.40392629e-03, -4.52577733e-02, -5.43994876e-03],
        [ 8.93737748e-03,  3.77042815e-02, -4.74891290e-02, ...,
         -2.90719271e-02,  8.57772119e-03,  2.14709882e-02]]),
 'documents': ['Virat Kohli is a world-renowned Indian international cricketer born on November 5, 1988, in Delhi. Considered one of the greatest batsmen in the history of the game, he is known for his aggressive style, exceptional run-chasing ability, and unwavering consistency, earning him the nickname "Chase Master" and "King Kohli".',
  'Mahendra Singh Dhoni, affectionately known as "Mahi" or "Captain Cool," is a legendary Indian cricketer and one of the most succ

In [29]:
vector_store.similarity_search(
    query="Who is referred as Hitman?",
    k=2,  # top_K similar objects are returned
)

[Document(id='8b48442d-20ff-473a-b39e-f76dacfa343c', metadata={'team': 'MI'}, page_content='Rohit Gurunath Sharma (born 30 April 1987) is a legendary Indian international cricketer and former captain of the national team in all formats. Popularly nicknamed "Hitman," he is widely recognized for his elegant batting style, prolific six-hitting ability, and calm, astute leadership.'),
 Document(id='ad701154-986a-4b18-a817-a93db0be28b1', metadata={'team': 'CSK'}, page_content='Mahendra Singh Dhoni, affectionately known as "Mahi" or "Captain Cool," is a legendary Indian cricketer and one of the most successful captains in international cricket history')]

In [31]:
vector_store.similarity_search_with_score(
    query="Who is greatest of all times?",
    k=2,
)

[(Document(id='ad701154-986a-4b18-a817-a93db0be28b1', metadata={'team': 'CSK'}, page_content='Mahendra Singh Dhoni, affectionately known as "Mahi" or "Captain Cool," is a legendary Indian cricketer and one of the most successful captains in international cricket history'),
  1.437252402305603),
 (Document(id='d85f78e3-c740-49fd-9fba-cc8bdaa40540', metadata={'team': 'RCB'}, page_content='Virat Kohli is a world-renowned Indian international cricketer born on November 5, 1988, in Delhi. Considered one of the greatest batsmen in the history of the game, he is known for his aggressive style, exceptional run-chasing ability, and unwavering consistency, earning him the nickname "Chase Master" and "King Kohli".'),
  1.4791696071624756)]

In [32]:
vector_store.similarity_search_with_score(
    query="",
    filter={"team": "CSK"}
)

[(Document(id='ad701154-986a-4b18-a817-a93db0be28b1', metadata={'team': 'CSK'}, page_content='Mahendra Singh Dhoni, affectionately known as "Mahi" or "Captain Cool," is a legendary Indian cricketer and one of the most successful captains in international cricket history'),
  1.8978285789489746)]

In [33]:
updated_doc1 = Document(
    page_content="Rohit sharma wifi is Ritika Sajdeh. He is a father of 1 daughter who lives in Mumbai India.",
    metadata={"team": "MI"}
)

In [34]:
vector_store.update_document(document_id="8b48442d-20ff-473a-b39e-f76dacfa343c", document=updated_doc1)

In [35]:
vector_store.get(include=["embeddings", "documents", "metadatas"])

{'ids': ['d85f78e3-c740-49fd-9fba-cc8bdaa40540',
  'ad701154-986a-4b18-a817-a93db0be28b1',
  '8b48442d-20ff-473a-b39e-f76dacfa343c'],
 'embeddings': array([[ 1.63408853e-02,  1.05008014e-01, -5.56412600e-02, ...,
         -3.30899097e-02,  4.41147499e-02,  7.48309249e-05],
        [-4.31474596e-02,  3.62290144e-02,  1.87813374e-03, ...,
         -6.40392629e-03, -4.52577733e-02, -5.43994876e-03],
        [-1.97886899e-02, -2.35153176e-02, -8.54668841e-02, ...,
          3.07120886e-02, -2.54571019e-03,  2.90817637e-02]]),
 'documents': ['Virat Kohli is a world-renowned Indian international cricketer born on November 5, 1988, in Delhi. Considered one of the greatest batsmen in the history of the game, he is known for his aggressive style, exceptional run-chasing ability, and unwavering consistency, earning him the nickname "Chase Master" and "King Kohli".',
  'Mahendra Singh Dhoni, affectionately known as "Mahi" or "Captain Cool," is a legendary Indian cricketer and one of the most succ

In [36]:
vector_store.similarity_search(
    query="Who is referred as Hitman?",
    k=2,  # top_K similar objects are returned
)

[Document(id='ad701154-986a-4b18-a817-a93db0be28b1', metadata={'team': 'CSK'}, page_content='Mahendra Singh Dhoni, affectionately known as "Mahi" or "Captain Cool," is a legendary Indian cricketer and one of the most successful captains in international cricket history'),
 Document(id='d85f78e3-c740-49fd-9fba-cc8bdaa40540', metadata={'team': 'RCB'}, page_content='Virat Kohli is a world-renowned Indian international cricketer born on November 5, 1988, in Delhi. Considered one of the greatest batsmen in the history of the game, he is known for his aggressive style, exceptional run-chasing ability, and unwavering consistency, earning him the nickname "Chase Master" and "King Kohli".')]

In [37]:
vector_store.delete(ids=["8b48442d-20ff-473a-b39e-f76dacfa343c"])